In [ ]:
# Copy_Atom 练习：
# 用 cute.make_copy_atom(cute.nvgpu.CopyUniversalOp(), cutlass.Float16, num_bits_per_copy=128) 
# 从 global memory 加载 128 个 fp16 到 shared memory，验证数据正确性。

import cutlass
import cutlass.cute as cute
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda 

n = 1000
block_size = 128
# smem_size = 256


@cute.kernel
def copy_atom_kernel(a:cute.Tensor,
                    b:cute.Tensor,
                    n,
                    block_size:cutlass.Constexpr[int]
                    ):

    copy_atom = cute.make_copy_atom(cute.nvgpu.CopyUniversalOp(), 
                                cutlass.Float16, 
                                num_bits_per_copy=128) 
    
    bidx,_,_  = cute.arch.block_idx()
    tidx,_,_  = cute.arch.thread_idx()
    idx = bidx*block_size +tidx

    # 按照block_size分块
    tile_a = cute.local_tile(a,(block_size,),(bidx,))
    tile_b = cute.local_tile(b,(block_size,),(bidx,))
    # tile_s = cute.local_tile(smem_tensor,(block_size,),(bidx))

    # block的共享内存大小为blocksize大小即可
    smem_layout = cute.make_layout((block_size,))
    # ★ 正确的 shared memory 分配方式
    smem_ptr = cute.arch.alloc_smem(cutlass.Float16, block_size)
    smem_tensor = cute.make_tensor(smem_ptr, smem_layout)
    cute.copy(copy_atom, tile_a, smem_tensor)
    #cute.copy(copy_atom,smem_tensor,tile_a)
    cute.arch.sync_threads()

    # thread视角,每thread4个
    # vec4_num = (block_size+3)//4
    # trd_s = cute.local_partition(smem_tensor,cute.make_layout((4,)),tidx)
    # trd_b = cute.local_partition(b_tile,cute.make_layout((4,)),tidx)
    # smem -> gmem
    cute.copy(copy_atom, smem_tensor, tile_b)
    # cute.arch.dealloc_smem(smem_ptr)

    # for k in cute.arange(0,4):
    #     trd_b[k] = trd_s[k] if tid+k<n else 0.

@cute.jit
def ex(a:cute.Tensor,
        b:cute.Tensor,
        n,
        stream:cuda.CUstream
        ):
        num_block = (n+block_size-1)//block_size
        copy_atom_kernel(a,b,n,block_size).launch(grid=(num_block,1,1),
                                                block=(block_size,1,1),
                                                stream=stream)
        

def demo():
    tensor = torch.randn(n,device='cuda',dtype=torch.float16)
    tensor_b = torch.zeros(n,device='cuda',dtype=torch.float16)
    a_tensor = from_dlpack(tensor).mark_layout_dynamic()
    b_tensor = from_dlpack(tensor_b).mark_layout_dynamic()
    
    stream = cuda.CUstream(0)
    compiled = cute.compile(ex,a_tensor,b_tensor,n,stream)
    compiled(a_tensor,b_tensor,n,stream)

    cuda.cuStreamSynchronize(stream)
    # 验证数据正确性
    assert torch.equal(tensor, tensor_b), "Copy 结果不一致!"
    print("验证通过, 前 8 个元素:", tensor_b.cpu().tolist()[:8])

demo()

验证通过, 前 8 个元素: [1.3740234375, 0.194580078125, 0.75634765625, 0.49365234375, -0.68505859375, -0.1929931640625, -0.057281494140625, 0.2296142578125]



## 6. 本篇练习
1. **跑通基线**：本地执行 `ampere/tensorop_gemm.py`，记录 TFLOPS 并验证正确性。
2. **调参分析**：修改 `BM, BN, BK` 和 stage 数，观察 TFLOPS 波动，理解寄存器溢出对性能的影响。
3. **Ncu 剖析**：用 `ncu` 抓取 GEMM kernel，关注：
   - `smsp__pipe_tensor_op_hmma_cycle_active`（MMA 阵列利用率）
   - `l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum`（访存压力）
4. **思考题**：为什么 `wgmma`/`tcgen05.mma` 强制 A 操作数从 SMEM 取，而不能像 Ampere `mma.sync` 那样全走寄存器？（提示：单条指令处理的数据量和寄存器容量的矛盾，以及 TMA 延迟隐藏需要"数据常驻 SMEM 供硬件直接寻址"这一前提）
5. **API 练习**：照着 §1、§2，把 `thr_mma.partition_A/B/C` + `make_fragment_A/B/C` 的完整调用链手写一遍，不看示例代码，检验是否记混了 `TiledMma` 和 `ThrMma` 的职责边界。


```bash
sudo $(which ncu) --kernel-name regex:kernel \
  --metrics \
smsp__pipe_tensor_op_hmma_cycles_active.avg.pct_of_peak_sustained_active,\
smsp__pipe_tensor_op_hmma_cycles_active.sum,\
l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,\
l1tex__t_sectors_pipe_lsu_mem_global_op_st.sum,\
l1tex__t_sectors_pipe_lsu_mem_local_op_ld.sum,\
launch__registers_per_thread,\
launch__shared_mem_per_block_dynamic \
  $(which python) ./tensorop_gemm.py \
  --mnkl 8192,8192,8192,1 --atom_layout_mnk 2,2,1 \
  --skip_ref_check --iterations 2
```


---
## 各 TODO 的量化验收总表（填完自检用）
假设 `--mnkl 256,256,256,1`，即 grid=(4,4,1)，共 16 个 CTA：
| TODO | 验收点 | 期望值 | 偏差时的诊断方向 |
|---|---|---|---|
| 0 | `num_threads` | 64 | 每个 atom 一个 warp × (2×2) |
| 1 | `print(tiled_mma)` 平铺形状 | (32, 32, 16) | permutation 的 `*2` 漏了 |
| 2 | ncu: `global_op_ld.sum` | **32768**（理论：16 CTAs × 8 k-tiles × (64+64)×32×2B ÷ 32B） | 大于理论值 → thread_layout 的 stride 写反；正好相等 → 合并访问满分 |
| 3b | grid | (4, 4, 1) | ceil_div 用错维度 |
| 5 | ncu: `shared_mem_per_block_dynamic` | 8.0 KB | (64+64)×32×2B |
| 7 | ncu: `registers_per_thread` | ~70–130 | >200 → fragment 挂错对象或 acc dtype 错 |
| 7 | ncu: `local_op_ld.sum` | 0 | 非 0 → 寄存器溢出，检查 (c) |
| 8 | 正确性 err | ≤ 2e-2 | 挂在这步最常见：`retile` 漏了 / transpose 标志错 |
| 9 | 正确性 + 每轮 sync 次数 | err ≤ 2e-2；每个 k_tile 恰好 2 次 sync | 主循环索引 `[None,None,kb]` 写成 `[kb]` 是高发错误 |
| 10 | ncu: `global_op_st.sum` | **远大于 4096**（理论最小 16×256=4096） | 若反而等于 4096 → 说明你聪明地绕过了反例，进阶实验 4 省了；记录实测/理论比值 |
**推荐 ncu 自检命令**（一条抓齐上表所有 device 侧指标）：
```bash
sudo $(which ncu) --kernel-name regex:kernel --launch-skip 10 --launch-count 1 \
  --metrics \
launch__registers_per_thread,\
launch__shared_mem_per_block_dynamic,\
l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,\
l1tex__t_sectors_pipe_lsu_mem_global_op_st.sum,\
l1tex__t_sectors_pipe_lsu_mem_local_op_ld.sum \
  $(which python) ./mma_copy_mini_todo.py --mnkl 256,256,256,1
```
---
## 卡住时的求助阶梯（按序尝试，禁止直接跳 4）
1. **读 TODO 里的思考题提示**——它们指向的正是关键机制；
2. **打印 host 侧对象**：`print(tiled_mma)` / `print(tiled_copy_A)` / `print(tCsA)`，CuTe 的 layout 会以可读形式打印，能直接看出分片形状对不对（这比看数字快得多）；
3. **二分定位**：把 TODO 9 的 MMA 部分临时替换成 `tCrC[None] = tCrA[None,None,0].load().sum()` 之类的占位，确认 G2S 数据通路正确，再恢复 MMA——分离“拷贝对不对”和“算的对不对”；
4. 最后才回 `tensorop_gemm.py` 对照，**对照完必须合上再默写一遍出错的那个 TODO**。
填完后，把你实测的 TFLOPS、`global_op_st` 实测/理论比、以及三道思考题（N 方向 `*2` 的原因、ldmatrix 无 bank conflict 的原因、retile 的必要性）的答案发给我，我来核对——思考题的答案质量比代码能否跑通更能说明你吃透了没。


In [ ]:
# mma_copy_mini_todo.py
import argparse
from typing import Tuple, Type
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

CTA_TILER = (64, 64, 32)
ATOM_LAYOUT = (2, 2, 1)
MMA_INST = (16, 8, 16)

class MmaCopyMini:
    def __init__(self, ab_dtype, c_dtype, acc_dtype):
        self.ab_dtype = ab_dtype
        self.c_dtype = c_dtype
        self.acc_dtype = acc_dtype
        self.cta_tiler = CTA_TILER
        self.atom_layout_mnk = ATOM_LAYOUT
        self.mma_inst_shape = MMA_INST
        self.num_threads = 128  # 2*2*32 = 128
        self.bM, self.bN, self.bK = self.cta_tiler

    # ==================== HOST 侧 ====================
    def _make_tiled_mma(self):
        op = cute.nvgpu.warp.MmaF16BF16Op(self.ab_dtype, self.acc_dtype, self.mma_inst_shape)
        mma_atom = cute.make_mma_atom(op)
        permutation_mnk = (
            self.atom_layout_mnk[0] * self.mma_inst_shape[0],
            self.atom_layout_mnk[1] * self.mma_inst_shape[1] * 2,
            self.atom_layout_mnk[2] * self.mma_inst_shape[2],
        )
        return cute.make_tiled_mma(mma_atom, cute.make_layout(self.atom_layout_mnk), permutation_mnk=permutation_mnk)

    def _make_tiled_copy_g2s(self, atom, tile_mn):
        dtype_bits = 16
        copy_elems = 128 // dtype_bits        # 8
        shape_count = tile_mn // copy_elems   # 64 // 8 = 8
        
        # ★ 绕过 stride= 关键字被忽略的 Bug：使用嵌套元组 (shape, stride) 作为单一参数
        # 等价于 cute.make_layout(shape=(8, 16), stride=(8, 64))
        thread_layout = cute.make_layout((
            (shape_count, self.num_threads // shape_count), # shape
            (copy_elems, tile_mn)                           # stride
        ))
        # 等价于 cute.make_layout(shape=(8, 1), stride=(1, 0))
        value_layout = cute.make_layout((
            (copy_elems, 1),  # shape
            (1, 0)             # stride
        ))
        
        return cute.make_tiled_copy_tv(atom, thread_layout, value_layout)

    @cute.jit
    def __call__(self, mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
        tiled_mma = self._make_tiled_mma()
        op = cute.nvgpu.CopyUniversalOp()
        atom_g2s_A = cute.make_copy_atom(op, self.ab_dtype, num_bits_per_copy=128)
        atom_g2s_B = cute.make_copy_atom(op, self.ab_dtype, num_bits_per_copy=128)
        tiled_copy_A = self._make_tiled_copy_g2s(atom_g2s_A, self.bM)
        tiled_copy_B = self._make_tiled_copy_g2s(atom_g2s_B, self.bN)
        grid = (cute.ceil_div(mA.shape[0], self.bM),
                cute.ceil_div(mB.shape[1], self.bN),
                1)
        self.kernel(mA, mB, mC, tiled_mma, tiled_copy_A, tiled_copy_B).launch(
            grid=grid, block=[self.num_threads, 1, 1]
        )

    # ==================== DEVICE 侧 ====================
    @cute.kernel
    def kernel(self, mA, mB, mC, tiled_mma, tiled_copy_A, tiled_copy_B):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()
        bM, bN, bK = self.bM, self.bN, self.bK

        # 4: CTA 级分块
        gA = cute.local_tile(mA, (bM, bK), (bidx, None))
        gB = cute.local_tile(mB, (bN, bK), (bidy, None))
        gC = cute.local_tile(mC, (bM, bN), (bidx, bidy))

        # 5: SMEM 布局 + 分配
        sA_layout = cute.make_layout((bM, bK))
        sB_layout = cute.make_layout((bN, bK))
        smem_allocator = cutlass.utils.SmemAllocator()
        sA = smem_allocator.allocate_tensor(self.ab_dtype, sA_layout)
        sB = smem_allocator.allocate_tensor(self.ab_dtype, sB_layout)

        # 6: G2S 线程级分区
        thr_copy_A = tiled_copy_A.get_slice(tidx)
        thr_copy_B = tiled_copy_B.get_slice(tidx)
        tAgA, tAsA = thr_copy_A.partition_S(gA), thr_copy_A.partition_D(sA)
        tBgB, tBsB = thr_copy_B.partition_S(gB), thr_copy_B.partition_D(sB)

        # 7: MMA 分区 + 寄存器 fragment
        thr_mma = tiled_mma.get_slice(tidx)
        tCsA = thr_mma.partition_A(sA)
        tCsB = thr_mma.partition_B(sB)
        tCgC = thr_mma.partition_C(gC)
        tCrA = tiled_mma.make_fragment_A(tCsA)
        tCrB = tiled_mma.make_fragment_B(tCsB)
        tCrC = tiled_mma.make_fragment_C(tCgC)
        tCrC.fill(0.0)

        # 8: S2R 拷贝（ldmatrix）+ retile
        atom_s2r = cute.make_copy_atom(
            cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=True, num_matrices=4),
            self.ab_dtype
        )
        tiled_s2r_A = cute.make_tiled_copy_A(atom_s2r, tiled_mma)
        tiled_s2r_B = cute.make_tiled_copy_B(atom_s2r, tiled_mma)
        thr_s2r_A = tiled_s2r_A.get_slice(tidx)
        thr_s2r_B = tiled_s2r_B.get_slice(tidx)
        tCsA_view = thr_s2r_A.partition_S(sA)
        tCsB_view = thr_s2r_B.partition_S(sB)
        tCrA_view = thr_s2r_A.retile(tCrA)
        tCrB_view = thr_s2r_B.retile(tCrB)

        k_tile_count = cute.size(tAgA, mode=[3])
        num_k_block = cute.size(tCrA, mode=[2])

        # 9: 主循环（朴素版：无流水线，双同步）
        for k in cutlass.range(k_tile_count):
            cute.copy(tiled_copy_A, tAgA[None, None, None, k], tAsA)
            cute.copy(tiled_copy_B, tBgB[None, None, None, k], tBsB)
            cute.arch.sync_threads()

            for kb in cutlass.range(num_k_block):
                cute.copy(tiled_s2r_A, tCsA_view[None, None, kb], tCrA_view[None, None, kb])
                cute.copy(tiled_s2r_B, tCsB_view[None, None, kb], tCrB_view[None, None, kb])
                cute.gemm(tiled_mma, tCrC, tCrA[None, None, kb], tCrB[None, None, kb], tCrC)

            cute.arch.sync_threads()

        # 10: Epilogue (反例：直写 GMEM)
        # 官方的正确写法（正例）：
        # RMEM -> SMEM：使用特殊的矩阵 store 指令（如 Blackwell 的 StMatrix 或 Ampere 的 cp.async），
        # 将寄存器中的结果写回一块专门布置好布局的 SMEM 中。这保证了合并访存。
        # SMEM -> GMEM：通过 TMA（Tensor Memory Access）或向量化 store 异步写回 GMEM。
        
        tCrD = cute.make_fragment_like(tCrC, self.c_dtype)
        tCrD[None] = tCrC.load().to(self.c_dtype)
        cute.autovec_copy(tCrD, tCgC)

# ======================================================================
def run(mnk: Tuple[int, int, int], iterations: int = 100, warmup: int = 10):
    import torch
    import cutlass.torch as cutlass_torch
    M, N, K = mnk
    assert M % 64 == 0 and N % 64 == 0 and K % 32 == 0
    def make(mode0, mode1, mode0_major, dtype):
        if mode0_major:
            t = (torch.empty(mode1, mode0, dtype=torch.int32).random_(-2, 2)
                 .to(dtype=cutlass_torch.dtype(dtype)).t().cuda())
        else:
            t = (torch.empty(mode0, mode1, dtype=torch.int32).random_(-2, 2)
                 .to(dtype=cutlass_torch.dtype(dtype)).cuda())
        return from_dlpack(t, assumed_align=16), t
    mA, a = make(M, K, True,  cutlass.Float16)
    mB, b = make(N, K, True,  cutlass.Float16)
    mC, c = make(M, N, False, cutlass.Float16)
    kernel = MmaCopyMini(cutlass.Float16, cutlass.Float16, cutlass.Float32)
    assert kernel.num_threads is not None, "TODO 0 未完成"
    compiled = cute.compile(kernel, mA, mB, mC)
    compiled(mA, mB, mC)
    torch.cuda.synchronize()
    ref = a.float() @ b.float().T
    err = (c.float() - ref).abs().max().item()
    status = "PASS" if err <= 2e-2 else "FAIL"
    print(f"[correctness] max_abs_err = {err:.5f}   {status}")
    if status == "FAIL":
        print("  → 检查顺序建议：TODO 9 主循环索引 → TODO 7 fragment → TODO 8 retile")
        return
    start, stop = torch.cuda.Event(True), torch.cuda.Event(True)
    for _ in range(warmup):
        compiled(mA, mB, mC)
    torch.cuda.synchronize()
    times = []
    for _ in range(iterations):
        start.record()
        compiled(mA, mB, mC)
        stop.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(stop))
    t = min(times) / 1e3
    print(f"[perf] t = {t*1e6:.2f} us | {2*M*N*K/t/1e12:.2f} TFLOPS | "
          f"{(M*K+N*K+M*N)*2/t/1e9:.1f} GB/s (effective)")

parser = argparse.ArgumentParser()
parser.add_argument("--mnkl", type=lambda s: tuple(int(x) for x in s.split(",")),
                    default=(256, 256, 256, 1))
parser.add_argument("--iterations", type=int, default=100)
args, _ = parser.parse_known_args()
run(args.mnkl[:3], args.iterations)


ValueError: Expected size in shape to be strictly positive, but got 0

In [15]:
# mma_copy_mini_todo.py
# 目标：手写 copy atom (G2S + ldmatrix S2R) + MMA atom (m16n8k16) 完整调用链
# 约束：M % 64 == 0, N % 64 == 0, K % 32 == 0；a_major=m, b_major=n, c_major=n
# 提示粒度：每个 TODO 给出 (1) 要做什么 (2) 可用的 API (3) 量化验收标准
#           想不起 API 签名时先试着自己默写，再回 tensorop_gemm.py 对照——对照不算作弊，
#           但必须先写完再看。
import argparse
from typing import Tuple, Type
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
# 固定的形状参数（本练习不改）
CTA_TILER = (64, 64, 32)        # (bM, bN, bK)
ATOM_LAYOUT = (2, 2, 1)         # mma atom 平铺方式 → 2x2x32 = 128? 不，是 2*2 个 warp? 算一下
MMA_INST = (16, 8, 16)          # m16n8k16
class MmaCopyMini:
    def __init__(self, ab_dtype, c_dtype, acc_dtype):
        self.ab_dtype = ab_dtype
        self.c_dtype = c_dtype
        self.acc_dtype = acc_dtype
        self.cta_tiler = CTA_TILER
        self.atom_layout_mnk = ATOM_LAYOUT
        self.mma_inst_shape = MMA_INST
        # 每block thread数?
        # 0（热身，一行）：算出 num_threads
        #   提示：每个 mma atom 由一个 warp 执行
        #   验收：应为 64（打印确认）
        self.num_threads = 128  # ← 替换这一行
        self.bM, self.bN, self.bK = self.cta_tiler
    # ==================== HOST 侧 ====================
    def _make_tiled_mma(self):
        # TODO 
        # 1: 构造 TiledMma
        #   步骤：
        #   (a) 创建 mma atom：cute.nvgpu.warp.MmaF16BF16Op(输入dtype, 累加dtype, 指令shape)
        #   (b) 计算 permutation_mnk = (atom_M*16, atom_N*8*2, atom_K*16)
        #       思考题（写在注释里，填完 step 7 你会回来验证）：
        #       为什么 N 方向要 *2？
        #   (c) cute.make_tiled_mma(op, atom_layout, permutation_mnk=...)
        #   验收：填完 TODO 4 后，kernel 能对 (64,64,32) 的 tile 正确覆盖
        op = cute.nvgpu.warp.MmaF16BF16Op(self.ab_dtype,self.acc_dtype,self.atom_layout_mnk)
        mma_atom = cute.make_mma_atom(op)
        return cute.make_tiled_mma(mma_atom,
                                   self.atom_layout_mnk,
                                   # (ATOM_LAYOUT[0]*16,ATOM_LAYOUT[1]*8*2,ATOM_LAYOUT[2]*16)
                                   )
        
    def _make_tiled_copy_g2s(self, atom, dtype, tile_mn):
        # 2: 构造 G2S 的 TiledCopy（thread-value 版本）
        #   背景：a_major=m / b_major=n → 连续维是 M/N，我们希望线程沿连续维排布
        #         这样一次 copy 指令覆盖 128bit = 8 个 fp16 的连续内存 → 完全合并访问
        #   步骤：
        #   (a) copy_elems = 128bit / 元素位宽          → 应为 8
        #   (b) shape_cont = tile_mn / copy_elems       → 连续维上的线程数，应为 8
        #   (c) thread_layout：形状 (shape_cont, num_threads/shape_cont)
        #       stride 要让**连续维方向上的相邻线程**对应**相邻地址**——自己推 stride
        #   (d) value_layout：形状 (copy_elems, 1)
        #   (e) cute.make_tiled_copy_tv(atom, thread_layout, value_layout)
        #   验收（ncu）：填完全部 TODO 后 global_op_ld.sum == 理论值（见文末表格）
        #               若实测 > 理论值 → 99% 是 (c) 的 stride 写反了
        copy_elems = 128/dtype # 每线程
        shape_count = tile_mn/copy_elems
        thread_layout = cute.make_layout((shape_count,self.num_threads/shape_count),
                                         stride=(tile_mn,copy_elems)) # num_threads
        value_layout = cute.make_layout((copy_elems,1)) # 每thread copy 数
        cute.make_tiled_copy_tv(atom,
                                thread_layout,
                                value_layout)

    @cute.jit
    def __call__(self, mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
        tiled_mma = self._make_tiled_mma()
        # 3a: 创建两个 G2S copy atom（A 和 B 各一个）
        #   提示：cute.make_copy_atom(op, dtype, num_bits_per_copy=128)
        #         op 用同步通用拷贝：cute.nvgpu.CopyUniversalOp()
        #   思考：这里为什么不用 cpasync.CopyG2SOp？（提示：本练习无流水线）
        op = cute.nvgpu.CopyUniversalOp(cute.nvgpu.CopyG2ROp)
        atom_g2s_A = cute.make_copy_atom(op,
                                         self.ab_dtype,
                                         num_bits_per_copy=128)
        atom_g2s_B = cute.make_copy_atom(op,
                                         self.ab_dtype,
                                         num_bits_per_copy=128)
        tiled_copy_A = self._make_tiled_copy_g2s(atom_g2s_A, mA.element_type, self.bM)
        tiled_copy_B = self._make_tiled_copy_g2s(atom_g2s_B, mB.element_type, self.bN)
        # 3b: 计算 grid 并 launch
        #   提示：grid 三个维度分别对应 C 分块的 (M, N, L)，
        #         用 cute.ceil_div(mC.shape[i], tile_i)
        #   验收：--mnkl 256,256,256,1 时 grid 应为 (4, 4, 1)
        grid = (cute.ceil_div(mA.shape[0],self.bM),
                cute.ceil_div(mB.shape[1],self.bN),
                1)
        self.kernel(mA, mB, mC, tiled_mma, tiled_copy_A, tiled_copy_B).launch(
            grid=grid, block=[self.num_threads, 1, 1]
        )
    # ==================== DEVICE 侧 ====================
    @cute.kernel
    def kernel(self, mA, mB, mC, tiled_mma, tiled_copy_A, tiled_copy_B):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()
        bM, bN, bK = self.bM, self.bN, self.bK
        # 4: CTA 级分块
        #   用 cute.local_tile 从 mA/mB/mC 中切出本 CTA 的 tile：
        #     gA: (bM, bK, k)   proj=(1, None, 1)
        #     gB: (bN, bK, k)   proj=(None, 1, 1)
        #     gC: (bM, bN)      proj=(1, 1, None)
        #   提示：coord=(bidx, bidy, None)
        #   验收：在 host 侧（__call__ 里 launch 前）print(tiled_mma) 和
        #         print(tiled_copy_A) 检查平铺形状是否为 (32,32,16) 和 (64,64,32)
        # ga、gb应该是一行/列，gc是一块？
        gA = cute.local_tile(mA,(bM,1),(bidx,bidy))
        gB = cute.local_tile(mB,(1,bN),(bidx,bidy))
        gC = cute.local_tile(mC,(bM,bN),(bidx,bidy))
        # 5: SMEM 布局 + 分配
        #   (a) sA_layout = cute.make_layout((bM, bK))     # col-major 型: (bM,bK):(1,bM)
        #       sB_layout 同理用 bN
        #       思考：这种布局下 ldmatrix 为什么天然无 bank conflict？
        #             （提示：ldmatrix 一次让 warp 内 32 线程各取一行 16B，行内连续 16B
        #              → 落在 4 个不同 bank 组。行主布局会怎样？→ 进阶实验 1 再验证）
        #   (b) 定义 @cute.struct SharedStorage：a、b 两个 MemRange，16B 对齐
        #   (c) cutlass.utils.SmemAllocator() 分配，用 .get_tensor(layout) 得到 sA/sB
        #   验收（ncu）：launch__shared_mem_per_block_dynamic == 8.0 KB
        sA_layout = cute.make_layout((bM, bK))     # col-major 型: (bM,bK):(1,bM)
        sB_layout = cute.make_layout((bN, bK))     # col-major 型: (bM,bK):(1,bM)
        sA = cutlass.utils.SmemAllocator().allocate_tensor(sA_layout)
        sB = cutlass.utils.SmemAllocator().allocate_tensor(sB_layout)
        # 6: G2S 线程级分区
        #   thr_copy_A = tiled_copy_A.get_slice(tidx)
        #   分出：tAgA = partition_S(gA)   # (CPY, CPY_M, CPY_K, k)  源=GMEM
        #         tAsA = partition_D(sA)   # (CPY, CPY_M, CPY_K)    目的=SMEM
        #   B 同理
        #   验收：num_k_tile = cute.size(tAgA, mode=[3])，K=256 时应为 8
        tild_A = tiled_copy_A.get_slice(tidx)
        tild_B = tiled_copy_B.get_slice(tidx)
        tAgA, tAsA = tild_A.partition_S(gA), tild_A.partition_D(sA)
        tBgB, tBsB = tild_B.partition_S(gB), tild_A.partition_D(sB)
        # 7: MMA 分区 + 寄存器 fragment —— 本练习核心，注意职责边界！
        #   (a) thr_mma = tiled_mma.get_slice(tidx)          # TiledMma → ThrMma
        #   (b) 分区（挂 ThrMma 上）：
        #         tCsA = partition_A(sA)   # (MMA, MMA_M, MMA_K)
        #         tCsB = partition_B(sB)
        #         tCgC = partition_C(gC)
        #   (c) 分配 fragment（挂 TiledMma 上，不是 thr_mma！）：
        #         tCrA = make_fragment_A(tCsA)
        #         tCrB = make_fragment_B(tCsB)
        #         tCrC = make_fragment_C(tCgC)  # 累加器
        #   (d) tCrC.fill(0.0)
        #   验收（ncu）：launch__registers_per_thread ≈ 100±30
        #               （fp32 累加器每线程 64 个 + fragment + 地址 + 杂项）
        #               若 > 200 → 检查是不是把 fragment 挂错了对象
        thr_mma = tiled_mma.get_slice(tidx)
        tCsA, tCsB, tCgC = thr_mma.partition_A(sA), thr_mma.partition(sB),thr_mma.partition(gC) 
        tCrA, tCrB, tCrC = thr_mma.make_fragment_A(tCsA), thr_mma.make_fragment_B(tCsB), thr_mma.make_fragment_C(tCgC)
        # 8: S2R 拷贝（ldmatrix）——最容易被忽略的 retile 在这里
        #   (a) 两个 copy atom：
        #         cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose, num_matrices)
        #         A: transpose=True   （a_major=m，col-major smem → 需要 transpose 语义）
        #         B: transpose=True
        #         num_matrices=4（与 tensorop_gemm.py 相同，一次 ldmatrix 取 4 个 8x8 矩阵）
        #   (b) 与 tiled_mma 绑定成 TiledCopy：
        #         tiled_s2r_A = cute.make_tiled_copy_A(atom, tiled_mma)
        #         tiled_s2r_B = cute.make_tiled_copy_B(atom, tiled_mma)
        #         思考：make_tiled_copy_A/B 和普通 make_tiled_copy_tv 的区别是什么？
        #               （提示：线程值布局必须与 mma fragment 的期望一致）
        #   (c) get_slice → partition_S(sA/sB) 得到 SMEM 视图
        #   (d) ★ retile ★：tCrA_view = thr_s2r_A.retile(tCrA)
        #         为什么必须 retile？（提示：tCrA 的寄存器布局是按 mma 的 TV layout 分的，
        #         而 ldmatrix 的写入布局不同，二者坐标系不一致）
        atom = cute.make_copy_atom(
            cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=1,num_matrices=4)
        )
        thr_s2r_A = cute.make_tiled_copy_A(atom,tiled_mma)
        thr_s2r_B = cute.make_tiled_copy_B(atom,tiled_mma)
        tids2rA = thr_s2r_A.get_slice(tidx)
        tids2rB = thr_s2r_B.get_slice(tidx)
        # tCsA = tids2rA.partition_S(sA)
        # tCsB = tids2rB.partition_S(sB)
        tCsA_view, tCrA_view = tids2rA.partition_S(sA), thr_s2r_A.retile(tCrA)
        tCsB_view, tCrB_view = tids2rB.partition_S(sB), thr_s2r_A.retile(tCrB)
        num_k_block = cute.size(tCrA, mode=[2])     # bK/16，K=32 时应为 2
        k_tile_count = cute.size(tAgA, mode=[3])    # K/bK，K=256 时应为 8
        # 9: 主循环（朴素版：无流水线，双同步）
        #   for k in cutlass.range(k_tile_count):
        #     (a) 内层循环 CPY_K 个 k 子块：cute.copy(tiled_copy_A, tAgA[..,ck,k], tAsA[..,ck])
        #         B 同理
        #     (b) cute.arch.sync_threads()    # 为什么需要？——所有 warp 写完 smem 才能读
        #     (c) 内层循环 num_k_block 个 k_block：
        #           ldmatrix：cute.copy(tiled_s2r_A, tCsA_view[..,kb], tCrA_view[..,kb])
        #           MMA：cute.gemm(tiled_mma, tCrC, tCrA[..,kb], tCrB[..,kb], tCrC)
        #     (d) 再一次 sync_threads()       # 为什么需要？——下一轮 G2S 不能覆盖还在读的 smem
        #   验收：正确性通过（见 run()）；随后数一数每个 k_tile 有几次同步、
        #         想想 tensorop_gemm.py 用什么手段干掉了它们（答案：register prefetch + 多 stage）

        # c的数据一直放外层？？
        for k in cutlass.range(k_tile_count):
            # g2s---这个不一样，后面的数就不一样
            cute.copy(tiled_copy_A, tAgA[k], tAsA[k])
            cute.copy(tiled_copy_B, tBgB[k], tBsB[k])
            cute.arch.sync_threads()

            # s2r
            cute.copy(atom,tCsA_view,tCrA_view)
            cute.copy(atom,tCsB_view,tCrB_view)
            cute.arch.sync_threads()

            # gemm
            cute.gemm(tiled_mma,tCrC,tCrA,tCrB,tCrC)
            cute.arch.sync_threads()

        # r2g:写回c
        cute.copy(atom,tCrC,tCgC) # ?


        # TODO 10: Epilogue（故意用"反例"写法）
        #   (a) tCrD = cute.make_fragment_like(tCrC, self.c_dtype)
        #   (b) 类型转换：tCrD[None] = tCrC.load().to(self.c_dtype)
        #   (c) 直写 GMEM：cute.autovec_copy(tCrD, tCgC)
        #   验收（ncu）：l1tex__t_sectors_pipe_lsu_mem_global_op_st.sum
        #         理论最小值 = 4096（见文末表格）；直写实测会是它的数倍
        #         → 这个"浪费"就是 tensorop_gemm.py 引入 sC staging 的全部理由
        # pass  # ← 解开注释并实现
# ======================================================================
# 以下全部是测试设施，不需要修改。你只填上面的 TODO 0-10。
# ======================================================================
def run(mnk: Tuple[int, int, int], iterations: int = 100, warmup: int = 10):
    import torch
    import cutlass.torch as cutlass_torch
    M, N, K = mnk
    assert M % 64 == 0 and N % 64 == 0 and K % 32 == 0
    def make(mode0, mode1, mode0_major, dtype):
        if mode0_major:
            t = (torch.empty(mode1, mode0, dtype=torch.int32).random_(-2, 2)
                 .to(dtype=cutlass_torch.dtype(dtype)).t().cuda())
        else:
            t = (torch.empty(mode0, mode1, dtype=torch.int32).random_(-2, 2)
                 .to(dtype=cutlass_torch.dtype(dtype)).cuda())
        return from_dlpack(t, assumed_align=16), t
    mA, a = make(M, K, True,  cutlass.Float16)   # A: M 连续 (a_major=m)
    mB, b = make(N, K, True,  cutlass.Float16)   # B: N 连续 (b_major=n)
    mC, c = make(M, N, False, cutlass.Float16)   # C: N 连续 (c_major=n)
    kernel = MmaCopyMini(cutlass.Float16, cutlass.Float16, cutlass.Float32)
    assert kernel.num_threads is not None, "TODO 0 未完成"
    compiled = cute.compile(kernel, mA, mB, mC)
    # 正确性
    compiled(mA, mB, mC)
    torch.cuda.synchronize()
    ref = a.float() @ b.float().T
    err = (c.float() - ref).abs().max().item()
    status = "PASS" if err <= 2e-2 else "FAIL"
    print(f"[correctness] max_abs_err = {err:.5f}   {status}")
    if status == "FAIL":
        print("  → 检查顺序建议：TODO 9 主循环索引 → TODO 7 fragment → TODO 8 retile")
        return
    # 性能
    start, stop = torch.cuda.Event(True), torch.cuda.Event(True)
    for _ in range(warmup):
        compiled(mA, mB, mC)
    torch.cuda.synchronize()
    times = []
    for _ in range(iterations):
        start.record()
        compiled(mA, mB, mC)
        stop.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(stop))
    t = min(times) / 1e3
    print(f"[perf] t = {t*1e6:.2f} us | {2*M*N*K/t/1e12:.2f} TFLOPS | "
          f"{(M*K+N*K+M*N)*2/t/1e9:.1f} GB/s (effective)")
    
# if __name__ == "__main__":
parser = argparse.ArgumentParser()
parser.add_argument("--mnkl", type=lambda s: tuple(int(x) for x in s.split(",")),
                    default=(256, 256, 256, 1))
parser.add_argument("--iterations", type=int, default=100)
args = parser.parse_args()
run(args.mnkl[:3], args.iterations)

usage: ipykernel_launcher.py [-h] [--mnkl MNKL] [--iterations ITERATIONS]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/1000/jupyter/runtime/kernel-v3833ee55b0507c5033b2e34f05e1f90e2f18a0e56.json


SystemExit: 2

要将当前的朴素版 GEMM 优化为**流水线版本**，核心思想是：**用计算掩盖访存延迟**。
在朴素版中，我们的循环是：`加载GMEM到SMEM -> 等待 -> S2R读取 -> MMA计算 -> 等待`。在这个过程中，Tensor Core 在等待数据加载时是闲置的。
流水线优化的核心机制是**多级缓冲**和**异步同步**。以下是将其改造为流水线版本的设计思路：
### 1. 核心机制：乒乓缓存
朴素版只有 1 块 SMEM 空间给 A 和 B。流水线需要分配多块（通常 2~8 块，称为 `num_stages`）。
假设 `stages=2`：
* 当 Tensor Core 正在从 `Buffer 0` 读取数据进行 MMA 计算时；
* 同时，CP.Async 硬件引擎正在把下一块数据从 GMEM 拉取到 `Buffer 1`。
### 2. 替换同步原语
朴素版使用 `cute.arch.sync_threads()`，这会让所有线程停下来等待。
流水线版需要引入异步屏障，例如 `cutlass.pipeline.PipelineTmaAsync` 或底层的 `mbarrier`。它允许：
* **Producer（生产者，执行拷贝的线程）**：拷贝完成后触发“到达”，但不阻塞。
* **Consumer（消费者，执行计算的线程）**：在计算前检查“数据是否到达”，到达则继续计算。
### 3. 循环结构的重构（软件流水线）
流水线的循环通常分为三个阶段：**起始（Prolog）**、**主体**、**收尾**。我们把朴素版的双层循环改造成下面这样：
```python
# 伪代码：展示流水线逻辑结构
# 【1. 起始阶段】预先发起第一块的 G2S 拷贝请求
k = 0
cute.copy(tiled_copy_A, tAgA[..., k], sA_pipe[0])
cute.copy(tiled_copy_B, tBgB[..., k], sB_pipe[0])
pipeline.producer_commit()  # 提交异步拷贝指令，不等待
# 【2. 主体阶段】计算当前块，同时预取下一块
for k in range(k_tile_count - 1):
    # 2.1 消费者：等待当前块 (stage % num_stages) 的数据到达 SMEM
    pipeline.consumer_wait(stage)
    
    # 2.2 生产者：预取下一块数据到下一个缓冲区
    cute.copy(tiled_copy_A, tAgA[..., k+1], sA_pipe[(stage+1) % num_stages])
    pipeline.producer_commit()
    
    # 2.3 内层循环：执行当前块的计算
    for kb in range(num_k_block):
        cute.copy(tiled_s2r_A, sA_pipe[stage][..., kb], tCrA_view[..., kb])
        cute.copy(tiled_s2r_B, sB_pipe[stage][..., kb], tCrB_view[..., kb])
        cute.gemm(tiled_mma, tCrC, tCrA[..., kb], tCrB[..., kb], tCrC)
        
    # 2.4 消费者：当前块计算完毕，释放该缓冲区，允许生产者覆盖它
    pipeline.consumer_release(stage)
    stage += 1
# 【3. 收尾阶段】处理最后一块数据
pipeline.consumer_wait(stage)
for kb in range(num_k_block):
    cute.copy(tiled_s2r_A, sA_pipe[stage][..., kb], tCrA_view[..., kb])
    cute.copy(tiled_s2r_B, sB_pipe[stage][..., kb], tCrB_view[..., kb])
    cute.gemm(tiled_mma, tCrC, tCrA[..., kb], tCrB[..., kb], tCrC)
pipeline.consumer_release(stage)
```
### 4. CuTe DSL 中的具体改造点
要把上面的逻辑落实到 CuTe DSL 代码中，你需要修改以下几个部分：
1. **SMEM 分配**
   不再分配单个 tensor，而是分配一个数组。SMEM 布局要增加一个维度：
   ```python
   # 之前的 SMEM 布局
   sA_layout = cute.make_layout((bM, bK))
   
   # 流水线版：增加 Stage 维度，假设 2 级
   num_stages = 2
   sA_layout_staged = cute.make_layout((bM, bK, num_stages))
   # sA[None, None, 0] 表示第 0 块
   ```
2. **引入 Pipeline 管理**
   你可以直接使用 CuTe 封装好的 `cutlass.pipeline`（正如你提供的官方 `dense_gemm.py` 中那样），或者使用更简单的 `mbarrier`。
   使用 `PipelineTmaAsync` 的好处是，它把所有的 wait / release 逻辑都封装好了。
3. **拷贝指令变为异步**
   朴素版用的是 `cute.copy(tiled_copy_A, ...)`，在底层会被编译为普通的 `LDG` 全局加载。
   流水线版需要换成 `cp.async`，在 CuTe DSL 中，这通常是通过修改 `CopyAtom` 实现的：
   ```python
   # 朴素版 CopyAtom
   op = cute.nvgpu.CopyUniversalOp()
   
   # 异步 CopyAtom (使用 cp.async)
   op = cute.nvgpu.cpasync.CopyG2SOp() 
   atom_g2s_A = cute.make_copy_atom(op, self.ab_dtype, num_bits_per_copy=128)
   ```
   这样 `cute.copy` 就会发出异步拷贝指令，执行流会立即往下走，不等待数据真正到达 SMEM。
### 总结
流水线优化的本质是**打破“加载-等待-计算”的串行关系**。在 CuTe DSL 中，你不需要写复杂的 PTX 汇编，只需要：
1. 把 SMEM 切成多块。
2. 挂载一个 `pipeline` 状态机。
3. 在循环中先发 `cp.async` 拷贝下一块，再 `wait` 拷贝当前块，再 `compute`，最后 `release`。
这样 GPU 就能在做 `gemm` 的时候，让 DMA 引擎自己去搬运下一块数据，从而让 TFLOPS 逼近硬件理论峰值。
